
# Unequal-sample permutation study — H0/H1, multiple distributions, paired imbalance ratios

This notebook is completely self-contained.

It studies unequal group sizes using

$$
N=n_1+n_2,
\qquad
r=\frac{n_1}{N},
$$

with several total sample sizes \(N\) and imbalance proportions \(r\).

For every scenario it:

- generates \(R\) paired datasets;
- evaluates a Monte Carlo permutation reference with \(B_{\mathrm{ref}}\) permutations;
- evaluates unequal-sample Ordered and Original EC Gaussian approximations;
- computes p-values, log-p-values, surprisal, and \(B_{\mathrm{eq}}\) in log-space;
- flags unresolved MC references when \(K=0\);
- aggregates \(B_{\mathrm{eq}}\) across \(R\) datasets as a ratio of sums;
- computes equivalent MC runtime;
- creates imbalance-gradient plots, unresolved-reference heatmaps, relative-\(B_{\mathrm{eq}}\) heatmaps, MSE diagnostics, and paired fraction-worse diagnostics.



## Cluster execution notes

This version is **OpenPBS-aware** and remains usable locally.

When a PBS job is detected:

- `PBS_O_WORKDIR` is treated as the project/submission directory.
- Persistent outputs are written to `/work/<user>/thesis_results/<run_name>/`.
- Frequent checkpoint I/O is performed in a job-specific `/scratch_local/...` directory.
- The checkpoint is mirrored to `/work` so another PBS job can resume safely.
- Matplotlib uses the non-interactive `Agg` backend.
- Every figure is saved automatically to the run's `figures/` directory.
- Numerical outputs and run metadata are saved automatically.
- The run folder includes a configuration hash, preventing accidental reuse of checkpoints from a different \(B_{\rm ref}\), \(R\), grid, or distribution configuration.

For the first cluster benchmark, use the same full settings as the local experiment (`FAST_SMOKE_TEST = False`).


In [ ]:

import os
import json
import math
import socket
import hashlib
import shutil
import re
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

# Detect PBS before importing pyplot. On the cluster we use a non-interactive
# backend so plotting works correctly in batch jobs.
IS_PBS = bool(os.environ.get("PBS_JOBID"))

import matplotlib
if IS_PBS:
    matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import gammaln, logsumexp
from scipy.stats import norm
from tqdm.auto import tqdm
from IPython.display import display

pd.set_option("display.max_columns", 100)

RUN_START = perf_counter()


In [ ]:

# ---------------- CONFIG ----------------
#
# The statistical settings below are unchanged from the local notebook.
# FAST_SMOKE_TEST=True is useful for a short interactive cluster test.
# For the benchmark run requested by the supervisor, set it to False.

FAST_SMOKE_TEST = False

if FAST_SMOKE_TEST:
    N_VALUES = [200, 500]
    RATIOS = [0.50, 0.70, 0.90]
    R = 10
    B_REF = 10_000
    BATCH_SIZE = 250
else:
    N_VALUES = [200, 500, 1_000, 2_000, 5_000]
    RATIOS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
    R = 10
    B_REF = 10_000
    BATCH_SIZE = 500

HYPOTHESES = ["H0", "H1"]

MEAN_X_H0 = 0.0
MEAN_Y_H0 = 0.0
MEAN_X_H1 = 0.0
MEAN_Y_H1 = 0.01

STANDARD_DEVIATION = 1.0
TAIL = "two-sided"
MIN_EXTREME_COUNT = 100
MASTER_SEED = 20260907

# ---------------- EXECUTION ENVIRONMENT ----------------

USER = os.environ.get("USER", "unknown")
PBS_JOB_ID = os.environ.get("PBS_JOBID", "local")
HOSTNAME = socket.gethostname()

# PBS_O_WORKDIR is the directory from which qsub was invoked.
# Locally, simply use the current working directory.
PROJECT_DIR = Path(
    os.environ.get("PBS_O_WORKDIR", Path.cwd())
).resolve()

print("Execution environment")
print("---------------------")
print("Mode:", "PBS cluster job" if IS_PBS else "local / interactive")
print("Host:", HOSTNAME)
print("PBS job ID:", PBS_JOB_ID)
print("Project directory:", PROJECT_DIR)

print("\nSimulation configuration")
print("------------------------")
print("N_VALUES:", N_VALUES)
print("RATIOS:", RATIOS)
print("R:", R, "B_REF:", B_REF, "BATCH_SIZE:", BATCH_SIZE)


In [ ]:

# ---------------- DISTRIBUTIONS ----------------

DISTRIBUTIONS = [
    {"label": "Gaussian", "family": "normal", "params": {}},
    {"label": "Exponential", "family": "exponential", "params": {}},
    {
        "label": "Gaussian mixture",
        "family": "gaussian_mixture",
        "params": {"component_mean": 1.5, "component_sd": 0.5},
    },
    {"label": "Gamma (shape=2)", "family": "gamma", "params": {"shape": 2.0}},
    {"label": "Student t (df=5)", "family": "student_t", "params": {"df": 5.0}},
    {"label": "Laplace", "family": "laplace", "params": {}},
]

def draw_standardized_sample(rng, size, family, params=None):
    params = {} if params is None else dict(params)
    family = family.lower()

    if family in {"normal", "gaussian"}:
        z = rng.normal(0, 1, size=size)

    elif family == "exponential":
        z = rng.exponential(scale=1.0, size=size) - 1.0

    elif family == "gamma":
        shape = float(params.get("shape", 2.0))
        raw = rng.gamma(shape=shape, scale=1.0, size=size)
        z = (raw - shape) / np.sqrt(shape)

    elif family in {"student_t", "student-t", "t"}:
        df = float(params.get("df", 5.0))
        if df <= 2:
            raise ValueError("Student-t df must be > 2.")
        raw = rng.standard_t(df, size=size)
        z = raw / np.sqrt(df / (df - 2.0))

    elif family == "laplace":
        z = rng.laplace(0.0, 1.0 / np.sqrt(2.0), size=size)

    elif family in {"gaussian_mixture", "normal_mixture", "mixture"}:
        mu = float(params.get("component_mean", 1.5))
        sigma = float(params.get("component_sd", 0.5))
        component = rng.integers(0, 2, size=size)
        loc = np.where(component == 0, -mu, mu)
        raw = rng.normal(loc=loc, scale=sigma, size=size)
        z = raw / np.sqrt(mu**2 + sigma**2)

    else:
        raise ValueError(f"Unknown family: {family}")

    return np.asarray(z, dtype=float)


# ---------------- RUN-SPECIFIC PATHS ----------------
#
# The hash makes each materially different experiment use a different folder,
# preventing stale checkpoints from a previous B_ref/R/grid from being reused.

RUN_CONFIG = {
    "fast_smoke_test": bool(FAST_SMOKE_TEST),
    "N_values": [int(x) for x in N_VALUES],
    "ratios": [float(x) for x in RATIOS],
    "R": int(R),
    "B_ref": int(B_REF),
    "batch_size": int(BATCH_SIZE),
    "hypotheses": list(HYPOTHESES),
    "mean_x_h0": float(MEAN_X_H0),
    "mean_y_h0": float(MEAN_Y_H0),
    "mean_x_h1": float(MEAN_X_H1),
    "mean_y_h1": float(MEAN_Y_H1),
    "standard_deviation": float(STANDARD_DEVIATION),
    "tail": TAIL,
    "min_extreme_count": int(MIN_EXTREME_COUNT),
    "master_seed": int(MASTER_SEED),
    "distributions": DISTRIBUTIONS,
}

RUN_HASH = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:10]

MODE_NAME = "smoke" if FAST_SMOKE_TEST else "full"
RUN_NAME = f"unbalanced_multidist_{MODE_NAME}_B{B_REF}_R{R}_{RUN_HASH}"

# Scrive nella DIRECTORY CORRENTE: submit_chain.sh (sul cluster) o le
# istruzioni del README (in locale) creano gia' una cartella contenitore
# univoca (nome_timestamp) e ci fanno cd DENTRO prima di eseguire questo
# script. Il notebook non deve occuparsi ne' di nome ne' di timestamp
# della propria cartella di output, solo scrivere relativo alla cwd.
OUTPUT_DIR = Path(".")
SCRATCH_DIR = OUTPUT_DIR / "scratch"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_RAW = OUTPUT_DIR / "raw_results_checkpoint.csv"

SUMMARY_FILE = OUTPUT_DIR / "scenario_summary.csv"
DIAGNOSTICS_FILE = OUTPUT_DIR / "diagnostics.csv"
PAIRED_FILE = OUTPUT_DIR / "paired_fraction_worse.csv"
METADATA_FILE = OUTPUT_DIR / "run_metadata.json"

RUN_METADATA = {
    "run_name": RUN_NAME,
    "run_hash": RUN_HASH,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "pbs" if IS_PBS else "local",
    "hostname": HOSTNAME,
    "user": USER,
    "pbs_job_id": PBS_JOB_ID,
    "pbs_o_workdir": os.environ.get("PBS_O_WORKDIR"),
    "project_dir": str(PROJECT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "scratch_dir": str(SCRATCH_DIR),
    "configuration": RUN_CONFIG,
}

with open(METADATA_FILE, "w") as f:
    json.dump(RUN_METADATA, f, indent=2)

print("\nRun paths")
print("---------")
print("Run name:", RUN_NAME)
print("Persistent output:", OUTPUT_DIR)
print("Scratch:", SCRATCH_DIR)
print("Figures:", FIGURE_DIR)


In [ ]:

# ---------------- PAIRED DATA GENERATION ACROSS RATIOS ----------------

def ratio_to_sizes(N, ratio):
    n1 = int(round(ratio * N))
    n2 = int(N - n1)
    if n1 < 2 or n2 < 2:
        raise ValueError(f"Invalid split N={N}, ratio={ratio}")
    return n1, n2

def generate_paired_ratio_datasets(
    N,
    ratios,
    hypothesis,
    data_seed,
    family,
    params=None,
):
    rng = np.random.default_rng(data_seed)

    max_n1 = max(ratio_to_sizes(N, r)[0] for r in ratios)
    max_n2 = max(ratio_to_sizes(N, r)[1] for r in ratios)

    base_x = draw_standardized_sample(rng, max_n1, family, params)
    base_y = draw_standardized_sample(rng, max_n2, family, params)

    if hypothesis == "H0":
        mx, my = MEAN_X_H0, MEAN_Y_H0
    else:
        mx, my = MEAN_X_H1, MEAN_Y_H1

    x_full = mx + STANDARD_DEVIATION * base_x
    y_full = my + STANDARD_DEVIATION * base_y

    out = {}
    for ratio in ratios:
        n1, n2 = ratio_to_sizes(N, ratio)
        out[float(ratio)] = (x_full[:n1].copy(), y_full[:n2].copy())

    return out


In [ ]:

# ---------------- MONTE CARLO ----------------

def difference_in_means(x, y):
    return float(np.mean(x) - np.mean(y))

def is_extreme(stats, observed, tail="two-sided"):
    if tail == "two-sided":
        return np.abs(stats) >= abs(observed)
    if tail == "greater":
        return stats >= observed
    if tail == "less":
        return stats <= observed
    raise ValueError("Invalid tail.")

def classify_mc_resolution(K, min_extreme_count=100):
    if K == 0:
        return "unresolved"
    if K < min_extreme_count:
        return "low_resolution"
    return "well_resolved"

def monte_carlo_permutation_test_unequal(
    x, y, B, rng, batch_size=500, tail="two-sided", min_extreme_count=100
):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    n1, n2 = len(x), len(y)
    N = n1 + n2

    pooled = np.concatenate([x, y])
    total_sum = float(pooled.sum())
    observed = difference_in_means(x, y)

    K = 0
    completed = 0
    elapsed = 0.0

    while completed < B:
        current = min(batch_size, B - completed)

        start = perf_counter()

        keys = rng.random((current, N))
        idx = np.argpartition(keys, kth=n1 - 1, axis=1)[:, :n1]

        s1 = pooled[idx].sum(axis=1)
        s2 = total_sum - s1

        stats = s1 / n1 - s2 / n2

        K += int(is_extreme(stats, observed, tail).sum())

        elapsed += perf_counter() - start
        completed += current

    if K > 0:
        log_p = float(np.log(K) - np.log(B))
        p = float(K / B)
        surprisal = float(-log_p / np.log(10.0))
    else:
        log_p = float("-inf")
        p = 0.0
        surprisal = float("nan")

    return {
        "p_value": p,
        "log_p_value": log_p,
        "surprisal": surprisal,
        "extreme_count": int(K),
        "resolution_status": classify_mc_resolution(K, min_extreme_count),
        "observed_statistic": observed,
        "elapsed_seconds": float(elapsed),
        "time_per_permutation": float(elapsed / B),
    }


In [ ]:

# ---------------- UNEQUAL EC APPROXIMATION ----------------

def unequal_ec_log_weights(n1, n2):
    k_min = max(0, n1 - n2)
    k_max = n1
    k = np.arange(k_min, k_max + 1, dtype=int)
    l = n1 - k

    log_c1 = (
        gammaln(n1 + 1)
        - gammaln(k + 1)
        - gammaln(n1 - k + 1)
    )
    log_c2 = (
        gammaln(n2 + 1)
        - gammaln(l + 1)
        - gammaln(n2 - l + 1)
    )

    lw = log_c1 + log_c2
    lw -= logsumexp(lw)

    return k, lw

def gaussian_class_log_tail_probability(obs, mu, var, tail="two-sided"):
    var = float(max(var, 0.0))

    if var == 0.0:
        if tail == "two-sided":
            event = abs(mu) >= abs(obs)
        elif tail == "greater":
            event = mu >= obs
        elif tail == "less":
            event = mu <= obs
        else:
            raise ValueError("Invalid tail.")
        return 0.0 if event else float("-inf")

    sd = np.sqrt(var)

    if tail == "two-sided":
        t = abs(obs)
        log_lower = norm.logcdf((-t - mu) / sd)
        log_upper = norm.logsf(( t - mu) / sd)
        out = float(np.logaddexp(log_lower, log_upper))
    elif tail == "greater":
        out = float(norm.logsf((obs - mu) / sd))
    elif tail == "less":
        out = float(norm.logcdf((obs - mu) / sd))
    else:
        raise ValueError("Invalid tail.")

    return min(out, 0.0)

def unequal_ec_gaussian_approximation_from_split(
    original_x, original_y, reference_a, reference_b, method_name, tail="two-sided"
):
    n1, n2 = len(original_x), len(original_y)
    N = n1 + n2

    observed = difference_in_means(original_x, original_y)

    k, log_weights = unequal_ec_log_weights(n1, n2)
    l = n1 - k

    mean_a = float(np.mean(reference_a))
    mean_b = float(np.mean(reference_b))
    var_a = float(np.var(reference_a, ddof=1))
    var_b = float(np.var(reference_b, ddof=1))

    class_means = (
        (N * k - n1**2) / (n1 * n2)
    ) * (mean_a - mean_b)

    scale = (N / (n1 * n2)) ** 2

    class_variances = scale * (
        (k * (n1 - k) / n1) * var_a
        +
        (l * (n2 - l) / n2) * var_b
    )

    class_log_tails = np.array(
        [
            gaussian_class_log_tail_probability(observed, mu, var, tail)
            for mu, var in zip(class_means, class_variances)
        ],
        dtype=float,
    )

    log_p = float(logsumexp(log_weights + class_log_tails))
    log_p = min(log_p, 0.0)

    return {
        "method": method_name,
        "log_p_value": log_p,
        "p_value": float(np.exp(log_p)),
        "surprisal": float(-log_p / np.log(10.0)),
    }

def ordered_ec_result_unequal(x, y, tail="two-sided"):
    n1 = len(x)
    pooled = np.sort(np.concatenate([x, y]))
    a = pooled[-n1:].copy()
    b = pooled[:-n1].copy()

    return unequal_ec_gaussian_approximation_from_split(
        x, y, a, b, "ordered", tail
    )

def original_ec_result_unequal(x, y, tail="two-sided"):
    return unequal_ec_gaussian_approximation_from_split(
        x, y, np.asarray(x).copy(), np.asarray(y).copy(), "original", tail
    )

UNEQUAL_EC_ESTIMATORS = {
    "ordered": ordered_ec_result_unequal,
    "original": original_ec_result_unequal,
}


In [ ]:

# ---------------- LOG-SPACE HELPERS ----------------

def log_abs_exp_difference(a, b):
    a, b = float(a), float(b)

    if np.isneginf(a) and np.isneginf(b):
        return float("-inf")
    if a == b:
        return float("-inf")

    hi, lo = max(a, b), min(a, b)

    if np.isneginf(lo):
        return hi

    return float(hi + np.log(-np.expm1(lo - hi)))

def log_p_one_minus_p(log_p):
    if np.isneginf(log_p):
        return float("-inf")
    if log_p >= 0.0:
        return float("-inf")

    return float(log_p + np.log(-np.expm1(log_p)))

def safe_exp(log_x):
    if np.isnan(log_x):
        return float("nan")
    if np.isposinf(log_x):
        return float("inf")
    if np.isneginf(log_x):
        return 0.0

    max_log = np.log(np.finfo(float).max)
    return float(np.exp(log_x)) if log_x < max_log else float("inf")

def deterministic_seed(*parts):
    ss = np.random.SeedSequence([int(p) for p in parts])
    return int(ss.generate_state(1, dtype=np.uint32)[0])


In [ ]:

# ---------------- RUNNER ----------------

def run_unbalanced_study(
    distributions,
    hypotheses,
    N_values,
    ratios,
    R,
    B_ref,
    batch_size,
    checkpoint_path=CHECKPOINT_RAW,
    resume=True,
):
    checkpoint_path = Path(checkpoint_path)

    if resume and checkpoint_path.exists():
        raw = pd.read_csv(checkpoint_path)
        print(f"Resuming from checkpoint with {len(raw):,} rows.")
    else:
        raw = pd.DataFrame()

    if raw.empty:
        done = set()
    else:
        counts = raw.groupby(
            ["hypothesis", "distribution", "run", "N", "ratio", "B_ref"]
        )["method"].nunique()

        done = {
            tuple(idx)
            for idx, count in counts.items()
            if count == len(UNEQUAL_EC_ESTIMATORS)
        }

    total = (
        len(hypotheses)
        * len(distributions)
        * len(N_values)
        * len(ratios)
        * R
    )

    pbar = tqdm(total=total, desc="Unequal-sample study", unit="scenario")
    pbar.update(len(done))

    new_rows = []

    try:
        for h_idx, hypothesis in enumerate(hypotheses):
            for d_idx, spec in enumerate(distributions):
                for N_idx, N in enumerate(N_values):
                    for run in range(R):

                        data_seed = deterministic_seed(
                            MASTER_SEED, 1000, h_idx, d_idx, N_idx, run
                        )

                        datasets = generate_paired_ratio_datasets(
                            N=N,
                            ratios=ratios,
                            hypothesis=hypothesis,
                            data_seed=data_seed,
                            family=spec["family"],
                            params=spec["params"],
                        )

                        added = False

                        for r_idx, ratio in enumerate(ratios):
                            key = (
                                hypothesis,
                                spec["label"],
                                run,
                                int(N),
                                float(ratio),
                            )

                            if key in done:
                                continue

                            x, y = datasets[float(ratio)]
                            n1, n2 = len(x), len(y)

                            mc_seed = deterministic_seed(
                                MASTER_SEED,
                                2000,
                                h_idx,
                                d_idx,
                                N_idx,
                                run,
                                r_idx,
                            )

                            mc = monte_carlo_permutation_test_unequal(
                                x=x,
                                y=y,
                                B=B_ref,
                                rng=np.random.default_rng(mc_seed),
                                batch_size=batch_size,
                                tail=TAIL,
                                min_extreme_count=MIN_EXTREME_COUNT,
                            )

                            for method, estimator in UNEQUAL_EC_ESTIMATORS.items():

                                start = perf_counter()
                                formula = estimator(x, y, tail=TAIL)
                                formula_time = perf_counter() - start

                                if mc["extreme_count"] == 0:
                                    log_num = float("-inf")
                                    log_sqerr = float("nan")
                                    status = "unresolved_reference"
                                else:
                                    log_num = log_p_one_minus_p(mc["log_p_value"])
                                    log_abs_err = log_abs_exp_difference(
                                        formula["log_p_value"],
                                        mc["log_p_value"],
                                    )
                                    log_sqerr = 2.0 * log_abs_err
                                    status = "resolved"

                                new_rows.append(
                                    {
                                        "hypothesis": hypothesis,
                                        "distribution": spec["label"],
                                        "family": spec["family"],
                                        "run": int(run),
                                        "N": int(N),
                                        "ratio": float(ratio),
                                        "n1": int(n1),
                                        "n2": int(n2),
                                        "method": method,
                                        "data_seed": int(data_seed),
                                        "mc_seed": int(mc_seed),
                                        "B_ref": int(B_ref),

                                        "p_ref": mc["p_value"],
                                        "log_p_ref": mc["log_p_value"],
                                        "surprisal_ref": mc["surprisal"],
                                        "mc_extreme_count": mc["extreme_count"],
                                        "mc_resolution_status": mc["resolution_status"],
                                        "mc_time_per_permutation": mc["time_per_permutation"],

                                        "p_formula": formula["p_value"],
                                        "log_p_formula": formula["log_p_value"],
                                        "surprisal_formula": formula["surprisal"],
                                        "formula_time_seconds": float(formula_time),

                                        "log_mc_numerator": log_num,
                                        "log_formula_squared_error": log_sqerr,
                                        "B_eq_status": status,
                                    }
                                )

                            done.add(key)
                            added = True
                            pbar.update(1)

                        if added and new_rows:
                            raw = pd.concat(
                                [raw, pd.DataFrame(new_rows)],
                                ignore_index=True,
                            )
                            raw.to_csv(checkpoint_path, index=False)

                            new_rows = []

    finally:
        pbar.close()

    return raw


In [ ]:

# ---------------- AGGREGATION ----------------

def summarize_unbalanced_study(raw):
    rows = []

    for key, group in raw.groupby(
        ["hypothesis", "distribution", "family", "N", "ratio", "n1", "n2", "method"],
        sort=True,
    ):
        hypothesis, distribution, family, N, ratio, n1, n2, method = key

        unresolved = int((group["mc_extreme_count"] == 0).sum())
        lowres = int((group["mc_resolution_status"] == "low_resolution").sum())
        well = int((group["mc_resolution_status"] == "well_resolved").sum())

        if unresolved > 0:
            log_B = float("nan")
            B_eq = float("nan")
            equiv_time = float("nan")
            status = "unresolved_reference"
        else:
            log_num = float(
                logsumexp(group["log_mc_numerator"].to_numpy(float))
            )
            log_den = float(
                logsumexp(group["log_formula_squared_error"].to_numpy(float))
            )

            log_B = (
                float("inf")
                if np.isneginf(log_den)
                else float(log_num - log_den)
            )

            B_eq = safe_exp(log_B)

            mc_cost = float(
                np.median(group["mc_time_per_permutation"].to_numpy(float))
            )

            equiv_time = (
                safe_exp(log_B + np.log(mc_cost))
                if np.isfinite(log_B) and mc_cost > 0
                else float("inf") if np.isposinf(log_B)
                else float("nan")
            )

            status = "resolved"

        rows.append(
            {
                "hypothesis": hypothesis,
                "distribution": distribution,
                "family": family,
                "N": int(N),
                "ratio": float(ratio),
                "n1": int(n1),
                "n2": int(n2),
                "method": method,
                "R": int(len(group)),
                "well_resolved_references": well,
                "low_resolution_references": lowres,
                "unresolved_references": unresolved,
                "B_eq_status": status,
                "log_aggregated_B_eq": log_B,
                "aggregated_B_eq": B_eq,
                "equivalent_mc_time_seconds": equiv_time,
                "median_formula_time_seconds": float(
                    np.median(group["formula_time_seconds"].to_numpy(float))
                ),
            }
        )

    return pd.DataFrame(rows)


In [ ]:

# ---------------- DIAGNOSTICS ----------------

def build_diagnostics(raw, summary):
    rows = []

    for key, group in raw.groupby(
        ["hypothesis", "distribution", "N", "ratio", "method"],
        sort=True,
    ):
        hypothesis, distribution, N, ratio, method = key

        logs = group["log_formula_squared_error"].to_numpy(float)
        valid = np.isfinite(logs)

        if valid.any():
            log_mse = float(logsumexp(logs[valid]) - np.log(valid.sum()))
            mse = safe_exp(log_mse)
        else:
            log_mse = float("nan")
            mse = float("nan")

        rows.append(
            {
                "hypothesis": hypothesis,
                "distribution": distribution,
                "N": int(N),
                "ratio": float(ratio),
                "method": method,
                "mean_formula_mse": mse,
                "log_mean_formula_mse": log_mse,
            }
        )

    diagnostics = pd.DataFrame(rows).merge(
        summary[
            ["hypothesis", "distribution", "N", "ratio", "method",
             "log_aggregated_B_eq", "aggregated_B_eq"]
        ],
        on=["hypothesis", "distribution", "N", "ratio", "method"],
        how="left",
    )

    diagnostics["relative_B_eq_to_balanced"] = np.nan
    diagnostics["log10_relative_B_eq_to_balanced"] = np.nan

    for _, g in diagnostics.groupby(
        ["hypothesis", "distribution", "N", "method"],
        sort=True,
    ):
        base = g.loc[np.isclose(g["ratio"], 0.5)]

        if base.empty:
            continue

        base_log = base["log_aggregated_B_eq"].iloc[0]

        if not np.isfinite(base_log):
            continue

        for idx in g.index:
            current = diagnostics.loc[idx, "log_aggregated_B_eq"]

            if not np.isfinite(current):
                continue

            delta = current - base_log

            diagnostics.loc[idx, "relative_B_eq_to_balanced"] = np.exp(delta)
            diagnostics.loc[idx, "log10_relative_B_eq_to_balanced"] = (
                delta / np.log(10.0)
            )

    return diagnostics

def build_fraction_worse(raw):
    work = raw.copy()
    work["formula_squared_error"] = np.nan

    valid = np.isfinite(work["log_formula_squared_error"])
    work.loc[valid, "formula_squared_error"] = np.exp(
        work.loc[valid, "log_formula_squared_error"]
    )

    rows = []

    for key, group in work.groupby(
        ["hypothesis", "distribution", "N", "method", "run"],
        sort=True,
    ):
        hypothesis, distribution, N, method, run = key

        base = group.loc[np.isclose(group["ratio"], 0.5)]
        if base.empty:
            continue

        baseline = base["formula_squared_error"].iloc[0]
        if not np.isfinite(baseline):
            continue

        for _, row in group.iterrows():
            current = row["formula_squared_error"]

            if not np.isfinite(current):
                continue

            rows.append(
                {
                    "hypothesis": hypothesis,
                    "distribution": distribution,
                    "N": int(N),
                    "method": method,
                    "run": int(run),
                    "ratio": float(row["ratio"]),
                    "imbalance_worse": current > baseline,
                }
            )

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .groupby(
            ["hypothesis", "distribution", "N", "method", "ratio"],
            as_index=False,
        )
        .agg(
            R_valid=("imbalance_worse", "size"),
            fraction_worse=("imbalance_worse", "mean"),
        )
    )


In [ ]:

# ---------------- PLOTTING ----------------

METHODS = ["ordered", "original"]

def safe_filename(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

def finish_figure(fig, filename):
    """
    Always save the figure to the persistent run directory.
    Locally, also display it. Under PBS, close it immediately.
    """
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")

    if IS_PBS:
        plt.close(fig)
    else:
        plt.show()

    return path



def ratio_gradient_colors(ratios):
    ratios = np.asarray(sorted(ratios), float)
    cmap = plt.get_cmap("Blues")

    scaled = (
        (ratios - ratios.min()) / (ratios.max() - ratios.min())
        if ratios.max() > ratios.min()
        else np.zeros_like(ratios)
    )

    return {
        float(r): cmap(0.30 + 0.65 * s)
        for r, s in zip(ratios, scaled)
    }

def posfinite(d, col):
    return np.isfinite(d[col]) & (d[col] > 0)

def plot_main_curves(summary, hypothesis):
    colors = ratio_gradient_colors(RATIOS)

    subset = summary.loc[summary["hypothesis"] == hypothesis]

    for distribution, dist_data in subset.groupby("distribution", sort=False):

        # Equivalent time
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

        for ax, method in zip(axes, METHODS):
            md = dist_data.loc[dist_data["method"] == method]

            any_valid = False

            for ratio in sorted(RATIOS):
                d = md.loc[np.isclose(md["ratio"], ratio)].sort_values("N")
                valid = posfinite(d, "equivalent_mc_time_seconds")

                if valid.any():
                    any_valid = True
                    ax.plot(
                        d.loc[valid, "N"],
                        d.loc[valid, "equivalent_mc_time_seconds"],
                        marker="o",
                        linewidth=2,
                        color=colors[float(ratio)],
                        label=f"r={ratio:.2f}",
                    )

            formula = (
                md.groupby("N", as_index=False)["median_formula_time_seconds"]
                .median()
                .sort_values("N")
            )

            vf = posfinite(formula, "median_formula_time_seconds")

            if vf.any():
                any_valid = True
                ax.plot(
                    formula.loc[vf, "N"],
                    formula.loc[vf, "median_formula_time_seconds"],
                    marker="s",
                    linestyle="--",
                    linewidth=3,
                    color="black",
                    label="Formula runtime",
                )

            if any_valid:
                ax.set_yscale("log")
            else:
                ax.text(
                    0.5, 0.5,
                    "No resolved equivalent-time values",
                    ha="center", va="center",
                    transform=ax.transAxes,
                )

            ax.set_xscale("log")
            ax.set_xlabel(r"Total sample size $N=n_1+n_2$")
            ax.set_title(method.capitalize())
            ax.grid(True, which="both", alpha=0.25)

        axes[0].set_ylabel("Equivalent MC runtime (seconds)")

        h, l = axes[-1].get_legend_handles_labels()
        fig.legend(h, l, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.04))

        fig.suptitle(
            f"{distribution} — {hypothesis}\nEffect of imbalance on equivalent MC runtime",
            y=1.11,
        )

        plt.tight_layout()
        finish_figure(
            fig,
            f"{safe_filename(distribution)}_{safe_filename(hypothesis)}_equivalent_time.png",
        )

        # B_eq
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

        for ax, method in zip(axes, METHODS):
            md = dist_data.loc[dist_data["method"] == method]
            any_valid = False

            for ratio in sorted(RATIOS):
                d = md.loc[np.isclose(md["ratio"], ratio)].sort_values("N")
                valid = posfinite(d, "aggregated_B_eq")

                if valid.any():
                    any_valid = True
                    ax.plot(
                        d.loc[valid, "N"],
                        d.loc[valid, "aggregated_B_eq"],
                        marker="o",
                        linewidth=2,
                        color=colors[float(ratio)],
                        label=f"r={ratio:.2f}",
                    )

            if any_valid:
                ax.set_yscale("log")
            else:
                ax.text(
                    0.5, 0.5,
                    "No resolved $B_{eq}$ values",
                    ha="center", va="center",
                    transform=ax.transAxes,
                )

            ax.set_xscale("log")
            ax.set_xlabel(r"Total sample size $N=n_1+n_2$")
            ax.set_title(method.capitalize())
            ax.grid(True, which="both", alpha=0.25)

        axes[0].set_ylabel("Aggregated equivalent MC permutations")

        h, l = axes[-1].get_legend_handles_labels()
        fig.legend(h, l, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.04))

        fig.suptitle(
            f"{distribution} — {hypothesis}\nEffect of imbalance on $B_{{eq}}$",
            y=1.11,
        )

        plt.tight_layout()
        finish_figure(
            fig,
            f"{safe_filename(distribution)}_{safe_filename(hypothesis)}_beq.png",
        )

def plot_unresolved_heatmaps(summary, hypothesis):
    subset = (
        summary.loc[summary["hypothesis"] == hypothesis]
        .groupby(["distribution", "N", "ratio"], as_index=False)
        ["unresolved_references"]
        .max()
    )

    for distribution, d in subset.groupby("distribution", sort=False):
        pivot = d.pivot(index="ratio", columns="N", values="unresolved_references")

        fig, ax = plt.subplots(figsize=(9, 6))
        im = ax.imshow(pivot.to_numpy(), aspect="auto", origin="lower")

        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels([str(int(x)) for x in pivot.columns])

        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels([f"{r:.2f}" for r in pivot.index])

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                v = pivot.iloc[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{int(v)}/{R}", ha="center", va="center")

        ax.set_xlabel("Total sample size N")
        ax.set_ylabel(r"Group-1 proportion $r=n_1/N$")
        ax.set_title(f"{distribution} — {hypothesis}\nUnresolved MC references (K=0)")

        plt.colorbar(im, ax=ax, label="Number unresolved")
        plt.tight_layout()
        finish_figure(
            fig,
            f"{safe_filename(distribution)}_{safe_filename(hypothesis)}_unresolved_heatmap.png",
        )

def plot_relative_Beq_heatmaps(diagnostics, hypothesis):
    subset = diagnostics.loc[diagnostics["hypothesis"] == hypothesis]

    for distribution, dist_data in subset.groupby("distribution", sort=False):
        for method in METHODS:
            d = dist_data.loc[dist_data["method"] == method]

            pivot = d.pivot(
                index="ratio",
                columns="N",
                values="log10_relative_B_eq_to_balanced",
            )

            fig, ax = plt.subplots(figsize=(9, 6))
            im = ax.imshow(pivot.to_numpy(), aspect="auto", origin="lower")

            ax.set_xticks(np.arange(len(pivot.columns)))
            ax.set_xticklabels([str(int(x)) for x in pivot.columns])

            ax.set_yticks(np.arange(len(pivot.index)))
            ax.set_yticklabels([f"{r:.2f}" for r in pivot.index])

            vals = pivot.to_numpy()

            for i in range(vals.shape[0]):
                for j in range(vals.shape[1]):
                    if np.isfinite(vals[i, j]):
                        ax.text(j, i, f"{vals[i,j]:.2f}", ha="center", va="center")

            ax.set_xlabel("Total sample size N")
            ax.set_ylabel(r"Group-1 proportion $r=n_1/N$")
            ax.set_title(
                f"{distribution} — {hypothesis} — {method.capitalize()}\n"
                r"$\log_{10}[B_{eq}(N,r)/B_{eq}(N,0.5)]$"
            )

            plt.colorbar(im, ax=ax, label="log10 relative B_eq")
            plt.tight_layout()
            finish_figure(
                fig,
                f"{safe_filename(distribution)}_{safe_filename(hypothesis)}_{safe_filename(method)}_relative_beq_heatmap.png",
            )

def plot_formula_mse(diagnostics, hypothesis):
    subset = diagnostics.loc[diagnostics["hypothesis"] == hypothesis]

    for distribution, dist_data in subset.groupby("distribution", sort=False):
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)

        for ax, method in zip(axes, METHODS):
            dm = dist_data.loc[dist_data["method"] == method]

            any_valid = False

            for N, d in dm.groupby("N", sort=True):
                d = d.sort_values("ratio")
                valid = posfinite(d, "mean_formula_mse")

                if valid.any():
                    any_valid = True
                    ax.plot(
                        d.loc[valid, "ratio"],
                        d.loc[valid, "mean_formula_mse"],
                        marker="o",
                        linewidth=2,
                        label=f"N={N}",
                    )

            if any_valid:
                ax.set_yscale("log")

            ax.set_xlabel(r"Group-1 proportion $r=n_1/N$")
            ax.set_title(method.capitalize())
            ax.grid(True, which="both", alpha=0.25)

        axes[0].set_ylabel("Average formula MSE")

        h, l = axes[-1].get_legend_handles_labels()
        fig.legend(h, l, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.03))

        fig.suptitle(
            f"{distribution} — {hypothesis}\nFormula error as imbalance increases",
            y=1.10,
        )

        plt.tight_layout()
        finish_figure(
            fig,
            f"{safe_filename(distribution)}_{safe_filename(hypothesis)}_formula_mse.png",
        )

def plot_fraction_worse(paired_summary, hypothesis):
    if paired_summary.empty:
        return

    subset = paired_summary.loc[paired_summary["hypothesis"] == hypothesis]

    for distribution, dist_data in subset.groupby("distribution", sort=False):
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)

        for ax, method in zip(axes, METHODS):
            dm = dist_data.loc[
                (dist_data["method"] == method)
                & (~np.isclose(dist_data["ratio"], 0.5))
            ]

            for N, d in dm.groupby("N", sort=True):
                d = d.sort_values("ratio")
                ax.plot(
                    d["ratio"],
                    d["fraction_worse"],
                    marker="o",
                    linewidth=2,
                    label=f"N={N}",
                )

            ax.axhline(0.5, linestyle="--", linewidth=1.5)
            ax.set_ylim(0, 1)
            ax.set_xlabel(r"Group-1 proportion $r=n_1/N$")
            ax.set_title(method.capitalize())
            ax.grid(True, alpha=0.25)

        axes[0].set_ylabel(
            "Fraction of paired datasets\nwhere imbalance increases squared error"
        )

        h, l = axes[-1].get_legend_handles_labels()
        fig.legend(h, l, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.03))

        fig.suptitle(
            f"{distribution} — {hypothesis}\n"
            "How consistently does imbalance worsen the approximation?",
            y=1.10,
        )

        plt.tight_layout()
        finish_figure(
            fig,
            f"{safe_filename(distribution)}_{safe_filename(hypothesis)}_fraction_worse.png",
        )


In [ ]:

# ---------------- RUN ----------------

STUDY_START = perf_counter()

raw_results = run_unbalanced_study(
    distributions=DISTRIBUTIONS,
    hypotheses=HYPOTHESES,
    N_values=N_VALUES,
    ratios=RATIOS,
    R=R,
    B_ref=B_REF,
    batch_size=BATCH_SIZE,
    checkpoint_path=CHECKPOINT_RAW,
    resume=True,
)

STUDY_RUNTIME_SECONDS = perf_counter() - STUDY_START

summary_results = summarize_unbalanced_study(raw_results)
summary_results.to_csv(SUMMARY_FILE, index=False)

diagnostics = build_diagnostics(raw_results, summary_results)
diagnostics.to_csv(DIAGNOSTICS_FILE, index=False)

paired_fraction_worse = build_fraction_worse(raw_results)
paired_fraction_worse.to_csv(PAIRED_FILE, index=False)

RUN_METADATA["study_runtime_seconds"] = float(STUDY_RUNTIME_SECONDS)
RUN_METADATA["study_runtime_minutes"] = float(STUDY_RUNTIME_SECONDS / 60.0)
RUN_METADATA["raw_rows"] = int(len(raw_results))
RUN_METADATA["summary_rows"] = int(len(summary_results))
RUN_METADATA["diagnostics_rows"] = int(len(diagnostics))
RUN_METADATA["paired_rows"] = int(len(paired_fraction_worse))

with open(METADATA_FILE, "w") as f:
    json.dump(RUN_METADATA, f, indent=2)

print(f"\nStudy runtime: {STUDY_RUNTIME_SECONDS:.2f} s "
      f"({STUDY_RUNTIME_SECONDS/60:.2f} min)")
print("Saved persistent numerical outputs to:", OUTPUT_DIR)

summary_view = summary_results[
    [
        "hypothesis",
        "distribution",
        "N",
        "ratio",
        "n1",
        "n2",
        "method",
        "R",
        "well_resolved_references",
        "low_resolution_references",
        "unresolved_references",
        "B_eq_status",
        "aggregated_B_eq",
        "equivalent_mc_time_seconds",
    ]
]

if IS_PBS:
    print(summary_view.to_string(index=False))
else:
    display(summary_view)


In [ ]:

# ---------------- PLOTS ----------------

for hypothesis in HYPOTHESES:
    print("\n" + "=" * 80)
    print(f"RESULTS: {hypothesis}")
    print("=" * 80)

    plot_main_curves(summary_results, hypothesis)
    plot_unresolved_heatmaps(summary_results, hypothesis)
    plot_relative_Beq_heatmaps(diagnostics, hypothesis)
    plot_formula_mse(diagnostics, hypothesis)
    plot_fraction_worse(paired_fraction_worse, hypothesis)


In [ ]:

# ---------------- SANITY CHECKS ----------------

def run_sanity_checks():
    n1, n2 = ratio_to_sizes(200, 0.5)
    assert n1 == 100 and n2 == 100

    for a, b in [(100, 100), (140, 60), (180, 20)]:
        _, lw = unequal_ec_log_weights(a, b)
        assert np.allclose(np.exp(lw).sum(), 1.0)

    p_ref = 0.2
    p_formula = 0.18

    direct = p_ref * (1 - p_ref) / (p_formula - p_ref) ** 2

    stable = np.exp(
        log_p_one_minus_p(np.log(p_ref))
        - 2 * log_abs_exp_difference(np.log(p_formula), np.log(p_ref))
    )

    assert np.allclose(direct, stable, rtol=1e-12)

    print("All sanity checks passed.")

run_sanity_checks()


# ---------------- FINALIZE RUN ----------------

TOTAL_RUNTIME_SECONDS = perf_counter() - RUN_START

RUN_METADATA["finished_utc"] = datetime.now(timezone.utc).isoformat()
RUN_METADATA["total_runtime_seconds"] = float(TOTAL_RUNTIME_SECONDS)
RUN_METADATA["total_runtime_minutes"] = float(TOTAL_RUNTIME_SECONDS / 60.0)
RUN_METADATA["completed_successfully"] = True

with open(METADATA_FILE, "w") as f:
    json.dump(RUN_METADATA, f, indent=2)

print(f"\nTotal notebook runtime: {TOTAL_RUNTIME_SECONDS:.2f} s "
      f"({TOTAL_RUNTIME_SECONDS/60:.2f} min)")
print("Persistent outputs:", OUTPUT_DIR)
print("Metadata:", METADATA_FILE)

# The notebook currently has no heavy temporary I/O, but a scratch
# directory is created for future extensions. Remove it only when empty.
if SCRATCH_DIR.exists():
    shutil.rmtree(SCRATCH_DIR, ignore_errors=True)
    print("Removed temporary scratch directory:", SCRATCH_DIR)
